# 05 — Regional & Channel Intelligence (Bonus)

Compute per-region and per-agent-type (EA vs IA) bind rates from the full dataset. These stats feed into Agent 4 for **dynamic escalation threshold adjustment** — regions with higher natural bind rates get different thresholds than underperforming regions.

Directly addresses Bonus Point 6 in the problem statement: *"Do EA agents outperform IA agents in conversion? Build the pipeline to dynamically adapt its thresholds per region and agent type."*

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

DATA_PATH = Path("../datasets/insurance_quotes.csv")
MODELS_DIR = Path("../models")

df = pd.read_csv(DATA_PATH)
df["Policy_Bind_enc"] = (df["Policy_Bind"] == "Yes").astype(int)
overall_bind_rate = df["Policy_Bind_enc"].mean()

print(f"Dataset: {df.shape[0]} rows")
print(f"Overall bind rate: {overall_bind_rate:.4f} ({overall_bind_rate*100:.1f}%)")
print(f"Regions: {sorted(df['Region'].unique())}")
print(f"Agent types: {sorted(df['Agent_Type'].unique())}")

## 1. Per-Region Bind Rates

In [ ]:
region_stats = df.groupby("Region").agg(
    total_quotes=("Quote_Num", "count"),
    bound_quotes=("Policy_Bind_enc", "sum"),
    bind_rate=("Policy_Bind_enc", "mean"),
    avg_premium=("Quoted_Premium", "mean"),
).reset_index()
region_stats["bind_rate_pct"] = (region_stats["bind_rate"] * 100).round(2)
region_stats = region_stats.sort_values("bind_rate", ascending=False)

print("=== Per-Region Statistics ===")
print(region_stats.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

region_stats.sort_values("bind_rate_pct").plot.barh(
    x="Region", y="bind_rate_pct", ax=axes[0], color="#3498db", legend=False
)
axes[0].axvline(x=overall_bind_rate * 100, color="red", linestyle="--", label=f"Overall ({overall_bind_rate*100:.1f}%)")
axes[0].set_xlabel("Bind Rate (%)")
axes[0].set_title("Bind Rate by Region")
axes[0].legend()

region_stats.sort_values("avg_premium").plot.barh(
    x="Region", y="avg_premium", ax=axes[1], color="#e67e22", legend=False
)
axes[1].set_xlabel("Average Premium ($)")
axes[1].set_title("Average Premium by Region")

plt.tight_layout()
plt.savefig(MODELS_DIR / "regional_bind_rates.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. EA vs IA Performance

In [ ]:
channel_stats = df.groupby("Agent_Type").agg(
    total_quotes=("Quote_Num", "count"),
    bound_quotes=("Policy_Bind_enc", "sum"),
    bind_rate=("Policy_Bind_enc", "mean"),
    avg_premium=("Quoted_Premium", "mean"),
).reset_index()
channel_stats["bind_rate_pct"] = (channel_stats["bind_rate"] * 100).round(2)

print("=== EA vs IA Performance ===")
print(channel_stats.to_string(index=False))

ea_rate = channel_stats.loc[channel_stats["Agent_Type"] == "EA", "bind_rate"].values[0]
ia_rate = channel_stats.loc[channel_stats["Agent_Type"] == "IA", "bind_rate"].values[0]
print(f"\nEA bind rate: {ea_rate*100:.2f}%")
print(f"IA bind rate: {ia_rate*100:.2f}%")
print(f"Difference:   {(ea_rate - ia_rate)*100:+.2f}pp")

## 3. Region x Agent Type Cross-Analysis

In [ ]:
cross_stats = df.groupby(["Region", "Agent_Type"]).agg(
    total_quotes=("Quote_Num", "count"),
    bound_quotes=("Policy_Bind_enc", "sum"),
    bind_rate=("Policy_Bind_enc", "mean"),
    avg_premium=("Quoted_Premium", "mean"),
).reset_index()
cross_stats["bind_rate_pct"] = (cross_stats["bind_rate"] * 100).round(2)

print("=== Region x Agent Type ===")
pivot = cross_stats.pivot_table(index="Region", columns="Agent_Type", values="bind_rate_pct")
print(pivot.to_string())

fig, ax = plt.subplots(figsize=(10, 6))
pivot.plot.bar(ax=ax, color=["#e67e22", "#1abc9c"])
ax.axhline(y=overall_bind_rate * 100, color="red", linestyle="--", label=f"Overall ({overall_bind_rate*100:.1f}%)")
ax.set_ylabel("Bind Rate (%)")
ax.set_title("EA vs IA Bind Rate by Region")
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(MODELS_DIR / "regional_ea_vs_ia.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Dynamic Threshold Computation

Agent 4 adjusts escalation thresholds per region. Regions with higher bind rates get **lower** auto-approve thresholds (more quotes auto-approved), and regions with lower bind rates get **higher** thresholds (more human review).

In [ ]:
BASE_AUTO_APPROVE_THRESHOLD = 75
SCALE_FACTOR = 50

def compute_dynamic_threshold(region_bind_rate, overall_rate, base=BASE_AUTO_APPROVE_THRESHOLD):
    adjustment = (region_bind_rate - overall_rate) * SCALE_FACTOR
    return round(base - adjustment, 1)

region_thresholds = {}
for _, row in region_stats.iterrows():
    region = row["Region"]
    rate = row["bind_rate"]
    threshold = compute_dynamic_threshold(rate, overall_bind_rate)
    region_thresholds[region] = {
        "bind_rate": round(rate, 4),
        "bind_rate_pct": round(rate * 100, 2),
        "auto_approve_threshold": threshold,
        "total_quotes": int(row["total_quotes"]),
        "avg_premium": round(row["avg_premium"], 2),
    }

channel_thresholds = {}
for _, row in channel_stats.iterrows():
    agent_type = row["Agent_Type"]
    rate = row["bind_rate"]
    threshold = compute_dynamic_threshold(rate, overall_bind_rate)
    channel_thresholds[agent_type] = {
        "bind_rate": round(rate, 4),
        "bind_rate_pct": round(rate * 100, 2),
        "auto_approve_threshold": threshold,
        "total_quotes": int(row["total_quotes"]),
        "avg_premium": round(row["avg_premium"], 2),
    }

print("=== Dynamic Thresholds by Region ===")
print(f"{'Region':<8} {'Bind Rate':>10} {'Threshold':>10} {'vs Base':>8}")
print("-" * 40)
for region in sorted(region_thresholds.keys()):
    info = region_thresholds[region]
    diff = info["auto_approve_threshold"] - BASE_AUTO_APPROVE_THRESHOLD
    print(f"{region:<8} {info['bind_rate_pct']:>9.2f}% {info['auto_approve_threshold']:>10.1f} {diff:>+7.1f}")

print(f"\n=== Dynamic Thresholds by Channel ===")
for agent_type, info in channel_thresholds.items():
    diff = info["auto_approve_threshold"] - BASE_AUTO_APPROVE_THRESHOLD
    print(f"{agent_type}: bind_rate={info['bind_rate_pct']:.2f}%, threshold={info['auto_approve_threshold']:.1f} ({diff:+.1f})")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
regions_sorted = sorted(region_thresholds.keys())
rates = [region_thresholds[r]["bind_rate_pct"] for r in regions_sorted]
thresholds = [region_thresholds[r]["auto_approve_threshold"] for r in regions_sorted]

x = np.arange(len(regions_sorted))
width = 0.35

bars1 = ax.bar(x - width / 2, rates, width, label="Bind Rate (%)", color="#3498db")
bars2 = ax.bar(x + width / 2, thresholds, width, label="Auto-Approve Threshold", color="#e74c3c", alpha=0.7)

ax.axhline(y=overall_bind_rate * 100, color="#3498db", linestyle="--", alpha=0.5)
ax.axhline(y=BASE_AUTO_APPROVE_THRESHOLD, color="#e74c3c", linestyle="--", alpha=0.5)

ax.set_xticks(x)
ax.set_xticklabels(regions_sorted)
ax.set_ylabel("Value")
ax.set_title("Regional Bind Rates vs Dynamic Escalation Thresholds")
ax.legend()
plt.tight_layout()
plt.savefig(MODELS_DIR / "regional_thresholds.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Save Regional Stats for Backend

In [ ]:
regional_intelligence = {
    "overall_bind_rate": round(overall_bind_rate, 4),
    "base_auto_approve_threshold": BASE_AUTO_APPROVE_THRESHOLD,
    "scale_factor": SCALE_FACTOR,
    "region_thresholds": region_thresholds,
    "channel_thresholds": channel_thresholds,
    "cross_stats": cross_stats.to_dict(orient="records"),
}

joblib.dump(regional_intelligence, MODELS_DIR / "regional_stats.joblib")
print("Saved: regional_stats.joblib")
print(f"\nContents: {list(regional_intelligence.keys())}")
print(f"Regions: {list(region_thresholds.keys())}")
print(f"Channels: {list(channel_thresholds.keys())}")